# What If Simulator
## Player Market Value Engine

In [15]:
import pandas as pd
import numpy as np
import sqlite3
import time
import random
import re
import json
import sys
import asyncio
import os

import threading

from rapidfuzz import process, fuzz

from playwright.async_api import async_playwright
import nest_asyncio

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
nest_asyncio.apply()


### **Library usage**

* **`playwright` & `nest_asyncio`**: `playwright` simulates a real browser to bypass Transfermarkt's anti-scraping blocks, while `nest_asyncio` prevents it from crashing Jupyter's event loop.
* **`rapidfuzz`**: Calculates string similarity (fuzzy matching) to accurately merge player profiles across our two datasets when their names don't match exactly.
* **`sqlite3`**: Stores our merged dataset in a fast, lightweight relational database to enable instant querying for the simulator UI.
* **`time` & `random`**: Injects randomized, human-like delays between our scraping requests to prevent rate limits and IP bans.
* **`re` (Regular Expressions)**: Cleans text data—like converting `"€120.00m"` into `120000000` so our XGBoost model receives clean numerical inputs.
* **`json`**: Quickly parses structured player data that modern websites frequently embed inside hidden `<script>` tags.

In [8]:
harvested_urls = []

def harvest_sofifa_urls_sync(target_count=7000):
    global harvested_urls
    player_urls = set()
    offset = 0
    base_url = "https://sofifa.com/players?offset="
    
    with sync_playwright() as p:
        # Keeping it visible (headless=False) so you can watch it work!
        browser = p.chromium.launch(headless=False) 
        page = browser.new_page()
        
        print(f"Initializing harvester for {target_count} players...")
        
        while len(player_urls) < target_count:
            current_url = f"{base_url}{offset}"
            
            try:
                # FIX: We changed 'networkidle' to 'domcontentloaded'. 
                # This stops waiting for ads/trackers and just looks at the HTML.
                page.goto(current_url, wait_until="domcontentloaded")
                
                # Wait directly for the player link to exist in the DOM
                page.wait_for_selector("a[href^='/player/']", timeout=15000)
                
                # Extract the profile links
                elements = page.query_selector_all("a[href^='/player/']")
                
                for el in elements:
                    href = el.get_attribute("href")
                    # Ensure it's a specific player profile, not a generic directory link
                    if href and len(href.split('/')) > 2: 
                        full_url = f"https://sofifa.com{href}"
                        player_urls.add(full_url)
                
                percentage = (len(player_urls) / target_count) * 100
                print(f"\r[Live Progress] Harvested: {len(player_urls)} / {target_count} URLs ({percentage:.1f}%)", end="", flush=True)
                
                if len(player_urls) >= target_count:
                    break
                    
                time.sleep(random.uniform(2.1, 4.5))
                offset += 60
                
            except Exception as e:
                print(f"\n[ERROR] Scraper stopped. The page title is currently: '{page.title()}'")
                print(f"Error Details: {e}")
                break 
            
        browser.close()
        harvested_urls = list(player_urls)[:target_count]

print("Starting harvester thread...")
scraper_thread = threading.Thread(target=harvest_sofifa_urls_sync, args=(7000,))
scraper_thread.start()
scraper_thread.join()

print(f"\n\nHarvest Complete! Total URLs ready for extraction: {len(harvested_urls)}")

Starting harvester thread...
Initializing harvester for 7000 players...
[Live Progress] Harvested: 7039 / 7000 URLs (100.6%)

Harvest Complete! Total URLs ready for extraction: 7000


In [13]:
# --- Cell 3: SoFIFA Profile Parser (Test Batch Targeting Article Tag) ---

# Defining the test batch 
test_urls = harvested_urls[:5]
scraped_players = []

def parse_sofifa_profiles(urls):
    global scraped_players
    
    with sync_playwright() as p:
        # Launching the visible browser to monitor behavior
        browser = p.chromium.launch(headless=False)
        page = browser.new_page()
        
        total_urls = len(urls)
        print(f"Starting test extraction for {total_urls} players...\n")
        
        for index, url in enumerate(urls, 1):
            try:
                # Loading the individual player profile
                page.goto(url, wait_until="domcontentloaded")
                page.wait_for_selector("h1", timeout=10000) 
                
                # Extracting the core identity
                name = page.locator("h1").first.inner_text()
                
                # Locating the main article tag which holds the actual player stats
                article_locator = page.locator("article").first
                
                # Extracting the entire text block of the player card
                raw_card_text = article_locator.inner_text() if article_locator.count() > 0 else "N/A"
                
                # Storing the raw data and replacing newlines with pipes for cleaner viewing
                player_data = {
                    "URL": url,
                    "Name": name,
                    "Raw_Card_Text": raw_card_text.replace('\n', ' | ')
                }
                
                scraped_players.append(player_data)
                
                # Updating the live progress tracker
                percentage = (index / total_urls) * 100
                progress_text = f"[Live Progress] Parsing: {index} / {total_urls} profiles ({percentage:.1f}%) | Last extracted: {name}"
                print(f"\r{progress_text.ljust(85)}", end="", flush=True)
                
            except Exception as e:
                print(f"\n[ERROR] Failing to extract data from {url}. Error: {e}")
            
            # Sleeping to mimic human behavior and avoid IP bans
            time.sleep(random.uniform(1.5, 3.5))
            
        browser.close()

# Running the parser thread
print("Starting parser thread...")
parser_thread = threading.Thread(target=parse_sofifa_profiles, args=(test_urls,))
parser_thread.start()
parser_thread.join()

# Displaying the clean DataFrame results
print("\n\n--- Test Batch Results ---")
df_test = pd.DataFrame(scraped_players)
print(df_test)

Starting parser thread...
Starting test extraction for 5 players...

[Live Progress] Parsing: 5 / 5 profiles (100.0%) | Last extracted: Nestor Jiménez    

--- Test Batch Results ---
                                                 URL            Name  \
0  https://sofifa.com/player/272465/caleb-roberts...   Caleb Roberts   
1        https://sofifa.com/player/random?1775801033     Liam Cooper   
2  https://sofifa.com/player/274482/adrian-ascues...   Adrián Ascues   
3  https://sofifa.com/player/272535/josh-bailey/2...     Josh Bailey   
4  https://sofifa.com/player/271880/nestor-jimene...  Nestor Jiménez   

                                       Raw_Card_Text  
0    | Caleb James Roberts |  |  CM CAM CDM | 19y...  
1    | Liam David Ian Cooper |  |   CB | 33y.o. (...  
2    | Adrián Ademir Ascues Earl |  |  CM CAM | 2...  
3    | Joshua Roy Bailey |  |  LWB | 18y.o. (Jun ...  
4    | Néstor Andrés Jiménez Ramírez |  |  ST RW ...  


In [ ]:
# Full SoFIFA Profile Extraction (7,000 Players) 

os.makedirs("data",exist_ok=True)   

# Storing the full dataset
full_scraped_players = []

# Defining the backup file path
backup_filepath = "data/sofifa_raw_data_backup.csv"

def parse_all_sofifa_profiles(urls):
    global full_scraped_players
    
    with sync_playwright() as p:
        # Launching visibly to mimic human behavior and avoid bot detection
        browser = p.chromium.launch(headless=False)
        page = browser.new_page()
        
        total_urls = len(urls)
        print(f"Starting full extraction for {total_urls} players.")
        
        for index, url in enumerate(urls, 1):
            try:
                # Loading the individual player profile
                page.goto(url, wait_until="domcontentloaded")
                page.wait_for_selector("h1", timeout=10000) 
                
                # Extracting the core identity
                name = page.locator("h1").first.inner_text()
                
                # Locating the main article tag 
                article_locator = page.locator("article").first
                
                # Extracting the entire text block of the player card
                raw_card_text = article_locator.inner_text() if article_locator.count() > 0 else "N/A"
                
                # Storing the raw data
                player_data = {
                    "URL": url,
                    "Name": name,
                    "Raw_Card_Text": raw_card_text.replace('\n', ' | ')
                }
                
                full_scraped_players.append(player_data)
                
                # Saving a backup to your local drive every 50 players to prevent data loss
                if index % 50 == 0:
                    pd.DataFrame(full_scraped_players).to_csv(backup_filepath, index=False)
                
                # Updating the live progress tracker
                percentage = (index / total_urls) * 100
                progress_text = f"[Live Progress] Parsing: {index} / {total_urls} profiles ({percentage:.1f}%) | Last extracted: {name}"
                print(f"\r{progress_text.ljust(85)}", end="", flush=True)
                
            except Exception as e:
                # Logging errors cleanly without stopping the entire loop
                print(f"\n[ERROR] Failing to extract data from {url}. Error: {e}")
            
            # Sleeping to mimic human behavior and avoid IP bans
            time.sleep(random.uniform(1.5, 3.5))
            
        browser.close()
        
        # Saving the final, complete dataset
        pd.DataFrame(full_scraped_players).to_csv("data/sofifa_raw_data_COMPLETE.csv", index=False)

# Running the parser thread on the full list of harvested URLs
print("Starting parser thread for the full batch...")
parser_thread = threading.Thread(target=parse_all_sofifa_profiles, args=(harvested_urls,))
parser_thread.start()

# Letting the thread run in the background 
parser_thread.join()

print("\n\nExtraction Complete! Data is safely saved to 'data/sofifa_raw_data_COMPLETE.csv'.")

Starting parser thread for the full batch...
Starting full extraction for 7000 players.
[Live Progress] Parsing: 49 / 7000 profiles (0.7%) | Last extracted: Ronald RodríguezClavelluza
[ERROR] Failing to extract data from https://sofifa.com/player/278265/adel-anzimati-aboudou/250044/. Error: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://sofifa.com/player/278265/adel-anzimati-aboudou/250044/", waiting until "domcontentloaded"

[Live Progress] Parsing: 7000 / 7000 profiles (100.0%) | Last extracted: Finlay Armstrong Lópezvesalvazasaa

Extraction Complete! Data is safely saved to 'sofifa_raw_data_COMPLETE.csv'.


In [18]:
df = pd.read_csv("data/sofifa_raw_data_COMPLETE.csv")
print(df['Raw_Card_Text'].iloc[0])

  | Caleb James Roberts |  |  CM CAM CDM | 19y.o. (Oct 24, 2005) 178cm 5'10" 74kg 163lbs |  | 56 | Overall rating | 	70 | Potential | 	€350K | Value | 	€600 | Wage |  9  1 Follow (12)  | History version (122) |   | Add to shortlist |  Customize Calculator | Profile |  | Preferred foot Right |  | 2  Skill moves |  | 3  Weak foot |  | 1  International reputation |  | Body type Lean (170-185) |  | Real face No |  | Release clause €831K |  | Acceleration type Controlled |  | 	 | Player specialities | 	 | Club |  |  Plymouth Argyle |  |  League One |  | 66  |  | Position RES |  | Kit number 24 |  | Joined Oct 24, 2022 |  | Contract valid until 2028 |  | 	 | Roles |  | Box-to-Box + |  | CM |  | Balanced | Ball-winning |  | 			 | Layout 1 2 3 | Attacking |  | 38 Crossing |  | 38 Finishing |  | 44 Heading accuracy |  | 61 Short passing |  | 38 Volleys |  | 	 | Skill |  | 59 Dribbling |  | 45 Curve |  | 37 FK Accuracy |  | 58 Long passing |  | 59 Ball control |  | 	 | Movement |  | 65 Accelerat

In [19]:
#Data Cleaning 
# Ensuring the data directory exists
os.makedirs("data", exist_ok=True)

# Load the raw dataset
df = pd.read_csv("data/sofifa_raw_data_COMPLETE.csv")

def extract_granular_stat(text, stat_name):
    pattern = r'\|\s*(\d{1,2})\s*' + re.escape(stat_name) + r'\s*\|'
    match = re.search(pattern, str(text), re.IGNORECASE)
    return int(match.group(1)) if match else None

def extract_core_rating(text, keyword):
    pattern = r'\|\s*(\d{1,2})\s*\|\s*' + re.escape(keyword) + r'\s*\|'
    match = re.search(pattern, str(text), re.IGNORECASE)
    return int(match.group(1)) if match else None

print("Parsing raw text blocks into precise behavioral features...")

# Extract 6 highly predictive granular stats instead of broad aggregates
df['Sprint_Speed'] = df['Raw_Card_Text'].apply(lambda x: extract_granular_stat(x, 'Sprint speed'))
df['Finishing'] = df['Raw_Card_Text'].apply(lambda x: extract_granular_stat(x, 'Finishing'))
df['Short_Passing'] = df['Raw_Card_Text'].apply(lambda x: extract_granular_stat(x, 'Short passing'))
df['Dribbling'] = df['Raw_Card_Text'].apply(lambda x: extract_granular_stat(x, 'Dribbling'))
df['Standing_Tackle'] = df['Raw_Card_Text'].apply(lambda x: extract_granular_stat(x, 'Standing tackle'))
df['Strength'] = df['Raw_Card_Text'].apply(lambda x: extract_granular_stat(x, 'Strength'))

# Extract Overall and Potential ratings
df['Overall'] = df['Raw_Card_Text'].apply(lambda x: extract_core_rating(x, 'Overall rating'))
df['Potential'] = df['Raw_Card_Text'].apply(lambda x: extract_core_rating(x, 'Potential'))

# Drop the messy raw column and any rows where the parser failed to find an Overall rating
df_clean = df.drop(columns=['Raw_Card_Text']).dropna(subset=['Overall', 'Sprint_Speed'])

# Save the final cleaned behavioral dataset
df_clean.to_csv("data/sofifa_cleaned_behavioral.csv", index=False)

print(f"Cleaning complete. Retained {len(df_clean)} players with fully structured behavioral profiles.")
display(df_clean.head())

Parsing raw text blocks into precise behavioral features...
Cleaning complete. Retained 6684 players with fully structured behavioral profiles.


,URL,Name,Sprint_Speed,Finishing,Short_Passing,Dribbling,Standing_Tackle,Strength,Overall,Potential
0,https://sofifa.com/player/272465/caleb-roberts...,Caleb Roberts,66.0,38.0,61.0,59.0,58.0,58.0,56.0,70.0
1,https://sofifa.com/player/random?1775801033,Javier Villar del Fraile,62.0,56.0,69.0,66.0,68.0,59.0,65.0,74.0
2,https://sofifa.com/player/274482/adrian-ascues...,Adrián Ascues,73.0,61.0,68.0,61.0,54.0,74.0,65.0,73.0
3,https://sofifa.com/player/272535/josh-bailey/2...,Josh Bailey,69.0,30.0,40.0,48.0,46.0,51.0,52.0,63.0
4,https://sofifa.com/player/271880/nestor-jimene...,Nestor Jiménez,59.0,67.0,60.0,66.0,20.0,52.0,61.0,70.0


### Feature Selection Rationale: Why We Dropped 80% of the Granular Stats

To predict market value effectively using XGBoost, we aggressively pruned the 40+ raw attributes down to 6-8 core features. Here is the technical rationale for this dimensionality reduction:

* **Preventing Multicollinearity:** Features like `Acceleration` and `Sprint Speed` (or `Standing Tackle` and `Sliding Tackle`) are highly correlated. Feeding both into tree-based models causes feature importance splitting, which ruins the SHAP values we need for model interpretability later. We retained single, strong proxies for each physical/technical domain.
* **Mitigating Overfitting (Curse of Dimensionality):** With a relatively small dataset (n ≈ 7,000), retaining 40+ features encourages the model to memorize specific stat combinations rather than learning generalized economic rules.
* **Optimizing Signal-to-Noise Ratio:** Granular attributes (e.g., `14 GK Diving`, `38 Volleys`, `FK Accuracy`) introduce mathematical noise. Market value is overwhelmingly driven by core competencies and aggregate ratings, not niche metrics. 
* **Simulator UI Constraints:** The final deliverable requires an interactive "What-If" simulator. Requiring end-users to adjust 40 distinct sliders to observe a price change is poor UX. Condensing to core features ensures a responsive, intuitive deployment.